#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType


#Reading data from Bronze

In [0]:
df = spark.table("databricks_lakehouse.bronze.erp_cust_az12")
df.display()

#Transformation

##Trimming

In [0]:
trim_plan = {
    field.name : F.trim(F.col(field.name))
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
}
df = df.withColumns(trim_plan)
df.display()

##NAS is to be removed from CID 

In [0]:
df = df.withColumn(
    "CID", F.regexp_replace(F.col('CID'), "^NAS", "")                           
)
df.display()

## checking for nulls

In [0]:
df.select([
    F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df.columns
]).display()

## Normalizing abbriviations

In [0]:
df = (
    df.withColumn(
        "GEN", 
        F.when(F.upper(F.col("GEN"))== "M", "Male")
        .when(F.upper(F.col("GEN"))== "F", "Female")
        .when(F.col("GEN").isNull(), "N/A")
        .otherwise(F.col("GEN"))
    )
)
df.display()

##Renaming the columns

In [0]:
rename_map = {
    "CID" : "customer_key",
    "BDATE" : "birth_date",
    "GEN" : "gender"
}

df = df.withColumnsRenamed(rename_map)
df.display()

#Writing data into silver table

In [0]:
(
    df.write.mode("overwrite")
    #.option("overwriteSchema", "true")  this incase we need to change the schema basically the column name of an existing table.
    .format("delta")
    .saveAsTable("databricks_lakehouse.silver.erp_customers")
)

#Checking the table

In [0]:
%sql
select * from databricks_lakehouse.silver.erp_customers limit 5